# MMTFv2 CL+GC Optuna Training (Fixed Mamba Backbone + Transformer VAE Conditioning)

Trains **MMTFAutoEncoderMamba** across CL, GC using Optuna hyperparameter optimization
with ProfitWeightedCE loss + BVS entropy regularization + Transformer VAE reconstruction/KL loss.

## Model: MMTFAutoEncoderMamba (ae_type="transformer_vae")
5-branch architecture with anti-collapse BVS + Transformer VAE regime conditioning:
1. **daily_fused**: Mamba backbone over full multimodal window
2. **summary_wind**: Dedicated LSTM over windowed summary (static-init)
3. **spatial**: Fused profile + raster (most recent day)
4. **sequential**: IntradayRNN on most recent day
5. **spatial_wind**: Attention over daily profile evolution

Anti-collapse: temperature scaling + entropy regularization + weight floor
Transformer VAE: Self-attention encoder/decoder over daily returns → KL-regularized latent

## AE Input
Daily returns computed from intraday.csv (5min OHLCV), anchored at 10:00 AM:
- `return_1d`: 10AM(T-1) → 10AM(T) pct change (×100, clipped ±20)
- `abs_return_1d`: Absolute return (vol proxy)
- `cumulative_5d_return`: Rolling 5-day cumulative log return

## Loss
```
total_loss = task_loss + recon_weight * (recon_loss + kl_weight * kl_loss) + entropy_loss
```

## Scoring
```
pnl_weight = (accuracy - baseline) if (accuracy - baseline) > 2.0
             else accuracy / baseline
score = avg_pnl * pnl_weight / loss_penalty
```

## 1. Environment Setup

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - skipping drive mount")

In [ ]:
# Add CTAFlow to path (Colab only)
if IN_COLAB:
    import sys
    %cd /content/drive/MyDrive/CTAEnv/
    %cd CTAFlow
    !git pull
    %cd ..
    sys.path.insert(0, '/content/drive/MyDrive/CTAEnv/CTAFlow/')
    sys.path.insert(1, '/content/drive/MyDrive/CTAEnv/SierraPy')
    !pip install -e SierraPy -q
    !pip install -e CTAFlow -q
    !pip install optuna -q
else:
    print("Running locally - ensure CTAFlow and optuna are installed")

In [ ]:
import json
import warnings
from datetime import date, time, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. Multi-Ticker Configuration

In [ ]:
# --- Tickers (CL + GC only) ---
TICKERS = ['CL', 'GC']

# --- Architecture (FIXED — not in Optuna search) ---
BACKBONE = 'mamba'

# --- Task ---
TASK = 'classification'
NUM_CLASSES = 3

# --- Paths ---
if IN_COLAB:
    DRIVE_PATH = Path('/content/drive/MyDrive')
    DATA_ROOT = DRIVE_PATH / 'features'
    RESULTS_PATH = DRIVE_PATH / 'results' / 'mmtfv2_cl_gc'
else:
    DATA_ROOT = Path('/workspace/model_data')
    RESULTS_PATH = Path('/workspace/results/mmtfv2_cl_gc')

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"Results path: {RESULTS_PATH}")
print(f"Tickers: {TICKERS}")
print(f"Backbone: {BACKBONE.upper()} (fixed)")

In [ ]:
# Verify data files for each ticker
print("\nChecking data files...")
required_files = ['features.csv', 'profiles.npz', 'vpin.parquet', 'rasterized.npz', 'target.csv']

all_found = True
for ticker in TICKERS:
    ticker_path = DATA_ROOT / ticker
    print(f"\n{ticker}:")
    for fname in required_files:
        fpath = ticker_path / fname
        status = "[OK]" if fpath.exists() else "[MISSING]"
        print(f"  {status} {fname}")
        if not fpath.exists():
            all_found = False

if not all_found:
    print("\n[WARNING] Some files missing - data loading may fail")

## 3. Load Data via TFTAlignedPrepLayer

In [ ]:
from CTAFlow.data.datasets.tft import (
    build_ticker_registry,
    summarize_event_routing,
)

# Show event routing for these tickers
print(summarize_event_routing(TICKERS))

In [ ]:
from pathlib import Path
from CTAFlow.models.prep.tft_aligned import TFTAlignedPrepLayer, quantile_classify
from CTAFlow.models.multi_asset import SummarySelectionConfig

# --- Step 1: Summary alignment config ---
summary_cfg = SummarySelectionConfig(strategy="exact")

# --- Step 2: Target transform ---
target_fn = quantile_classify(n_classes=NUM_CLASSES, expanding_min=60)

# --- Step 3: Load all tickers + align summaries + scale ---
print("Loading ticker data...")
prep = TFTAlignedPrepLayer.from_directories(
    root_dir=DATA_ROOT,
    tickers=TICKERS,
    window_size=10,              # Overridden per trial
    summary_config=summary_cfg,
    target_transform=target_fn,
    intraday_file="intraday.csv",      # Load 5min OHLCV for VAE daily returns
    intraday_target_time="10:00",      # 10AM-to-10AM returns
)

# --- Step 4: Scale all modalities (non-forward-looking) ---
print("\nScaling features...")
prep.scale_features()
print("  Summary:    rolling z-score / fixed multipliers")
print("  Sequential: (val-close)/close*100 + orderflow multipliers")
print("  Profiles:   channel-wise (vol÷15, price×100)")

# Inspect loaded dimensions
dims = prep.get_feature_dims()
print(f"\nFeature dimensions: {dims}")
print(f"n_tickers: {prep.n_tickers}")
print(f"n_asset_classes: {prep.n_asset_classes}")
print(f"n_asset_subclasses: {prep.n_asset_subclasses}")

# Check daily returns are loaded
for t in TICKERS:
    n_ret = len(prep._daily_returns.get(t, {}))
    print(f"  [{t}] Daily returns dates: {n_ret}")

In [ ]:
# Visualize target distribution per ticker
fig, axes = plt.subplots(1, len(TICKERS), figsize=(6*len(TICKERS), 5))
if len(TICKERS) == 1:
    axes = [axes]

for ticker, ax in zip(TICKERS, axes):
    targets = list(prep._targets.get(ticker, {}).values())
    if not targets:
        ax.set_title(f'{ticker}: No targets')
        continue
    target_arr = np.array(targets)
    if np.issubdtype(target_arr.dtype, np.integer) or len(np.unique(target_arr)) <= 5:
        classes, counts = np.unique(target_arr.astype(int), return_counts=True)
        labels = [f'Class {c}' for c in classes]
        colors = ['#e74c3c', '#95a5a6', '#27ae60'][:len(classes)]
        bars = ax.bar(labels, counts, color=colors)
        for bar, count in zip(bars, counts):
            pct = count / len(target_arr) * 100
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    f'{count}\n({pct:.1f}%)', ha='center', va='bottom')
    else:
        ax.hist(target_arr, bins=50, alpha=0.7)
    ax.set_ylabel('Count')
    ax.set_title(f'{ticker} Target Distribution')

plt.tight_layout()
plt.show()

## 4. Define Optuna Objective

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.tft.auto_mmtft import (
    MMTFAutoEncoderMamba,
    train_step_with_ae,
)
from CTAFlow.models.deep_learning.training.loss import ProfitWeightedCE
from CTAFlow.data.datasets.tft import unpack_batch_for_model

# Feature dimensions from loaded data
F_SUM = dims.get('f_sum', 20)
F_PROFILE = dims.get('f_profile', 3)
F_RASTER = dims.get('f_raster', 3)
F_SEQ = dims.get('f_seq', 10)
F_AE = 3  # [return_1d, abs_return_1d, cumulative_5d_return]

print(f"Model input dimensions:")
print(f"  f_sum={F_SUM}, f_profile={F_PROFILE}, f_raster={F_RASTER}")
print(f"  f_seq={F_SEQ}, f_ae={F_AE} (daily returns from intraday 10AM close)")
print(f"  n_tickers={prep.n_tickers}, n_classes={prep.n_asset_classes}, n_subclasses={prep.n_asset_subclasses}")
print(f"\nBackbone: {BACKBONE.upper()} (fixed — not in Optuna search)")
print(f"Architecture: MMTFAutoEncoder — 5 branches + anti-collapse BVS + Transformer VAE conditioning")

In [ ]:
def create_dataloaders(window_size: int, batch_size: int, val_cutoff_date=None, val_split=0.2):
    """Create TFT-aligned train/val dataloaders."""
    prep.window_size = window_size

    train_loader, val_loader = prep.get_loaders(
        val_cutoff_date=val_cutoff_date,
        val_ratio=val_split,
        batch_size=batch_size,
        num_workers=0,
        tickers=TICKERS,
        shuffle_train=True,
    )
    return train_loader, val_loader


def train_epoch_vae(model, loader, criterion, optimizer, device, max_norm=1.0,
                    recon_weight=None):
    """Train for one epoch WITH VAE recon/KL loss + BVS entropy regularization.

    Loss = task_loss + recon_weight * total_ae_loss + entropy_loss
    where total_ae_loss = recon_loss + kl_weight * kl_loss (VAE handles internally)
    """
    model.train()
    total_loss = 0.0
    total_task = 0.0
    total_recon = 0.0
    total_kl = 0.0
    total_entropy = 0.0
    total_correct = 0
    total_samples = 0

    rw = recon_weight if recon_weight is not None else model.recon_weight

    for batch in loader:
        inputs, targets = unpack_batch_for_model(batch, device=device)
        targets = targets.long()

        optimizer.zero_grad()
        logits, ae_losses = model(**inputs, return_ae_losses=True)

        task_loss = criterion(logits, targets)
        ae_loss = ae_losses['total_ae_loss']
        entropy_loss = model.get_entropy_loss()
        loss = task_loss + rw * ae_loss + entropy_loss

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        optimizer.step()

        total_loss += loss.item()
        total_task += task_loss.item()
        total_recon += ae_losses['recon_loss'].item()
        if 'kl_loss' in ae_losses:
            total_kl += ae_losses['kl_loss'].item()
        total_entropy += entropy_loss.item() if torch.is_tensor(entropy_loss) else entropy_loss
        total_samples += targets.size(0)
        _, predicted = logits.max(1)
        total_correct += predicted.eq(targets).sum().item()

    n = max(len(loader), 1)
    avg_loss = total_loss / n
    acc = 100.0 * total_correct / max(total_samples, 1)
    metrics = {
        'loss': avg_loss,
        'task_loss': total_task / n,
        'recon_loss': total_recon / n,
        'kl_loss': total_kl / n,
        'entropy_loss': total_entropy / n,
    }
    return avg_loss, acc, metrics


def evaluate(model, loader, criterion, device, transaction_cost=0.003):
    """Evaluate model on validation set."""
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    total_pnl = 0.0
    dir_correct = 0
    dir_attempts = 0
    total_recon = 0.0
    total_kl = 0.0

    with torch.no_grad():
        for batch in loader:
            inputs, targets = unpack_batch_for_model(batch, device=device)
            targets = targets.long()

            logits, ae_losses = model(**inputs, return_ae_losses=True)

            loss = criterion(logits, targets)
            total_loss += loss.item()
            total_recon += ae_losses['recon_loss'].item()
            if 'kl_loss' in ae_losses:
                total_kl += ae_losses['kl_loss'].item()
            total_samples += targets.size(0)

            _, predicted = logits.max(1)
            total_correct += predicted.eq(targets).sum().item()

            position = (predicted.float() - 1.0)
            true_dir = (targets.float() - 1.0)
            pnl = position * true_dir - transaction_cost * position.abs()
            total_pnl += pnl.sum().item()

            is_trade = (predicted != 1)
            if is_trade.any():
                dir_attempts += is_trade.sum().item()
                correct_dir = ((predicted == 2) & (targets == 2)) | ((predicted == 0) & (targets == 0))
                dir_correct += (correct_dir & is_trade).sum().item()

    n = max(len(loader), 1)
    avg_loss = total_loss / n
    acc = 100.0 * total_correct / max(total_samples, 1)
    avg_pnl = total_pnl / max(total_samples, 1)
    dir_acc = 100.0 * dir_correct / max(dir_attempts, 1)
    avg_recon = total_recon / n
    avg_kl = total_kl / n

    return avg_loss, avg_pnl, acc, dir_acc, avg_recon, avg_kl

In [ ]:
import math

def get_composite_score(avg_pnl, accuracy, val_loss, baseline=34.0, loss_dampen=0.5):
    """Composite scoring: Avg PnL weighted by accuracy edge, penalized by loss."""
    accuracy = max(accuracy, 1.01)
    baseline = max(baseline, 1.01)
    edge = accuracy - baseline
    pnl_weight = edge if edge > 2.0 else (accuracy / baseline)
    raw_score = avg_pnl * pnl_weight

    loss_scaled = val_loss * 10.0 if val_loss < 1.0 else val_loss
    loss_penalty = max(min(loss_scaled * loss_dampen, 10.0), 1.0)
    return raw_score / loss_penalty


def objective(trial: optuna.Trial) -> float:
    # --- Architecture (backbone FIXED to mamba) ---
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    grn_dropout = trial.suggest_float('grn_dropout', 0.05, 0.5)
    window_size = trial.suggest_int('window_size', 3, 15, step=3)
    d_model = trial.suggest_categorical('d_model', [64, 128])
    d_static_emb = trial.suggest_categorical('d_static_emb', [32, 64])
    n_attn_heads = trial.suggest_categorical('n_attn_heads', [2, 4, 8])

    # Mamba-specific (fixed backbone)
    d_state = trial.suggest_categorical('d_state', [16, 32])
    d_conv = trial.suggest_categorical('d_conv', [2, 4])
    expand = trial.suggest_categorical('expand', [1, 2])
    n_layers = trial.suggest_int('n_layers', 1, 2)

    # --- Training ---
    batch_size = trial.suggest_categorical('batch_size', [48, 64, 96])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 7e-3, log=True)
    max_norm = trial.suggest_float('max_norm', 0.5, 1.0)

    # --- ProfitWeightedCE ---
    profit_scale = trial.suggest_float('profit_scale', 10.0, 30.0, step=2.5)
    direction_penalty = trial.suggest_float('direction_penalty', 2.0, 2.5)
    min_weight = trial.suggest_float('min_weight', 0.5, 1.0)
    max_weight = trial.suggest_float('max_weight', 6.0, 15.0)

    # --- Anti-collapse BVS params ---
    bvs_temperature = trial.suggest_float('bvs_temperature', 1.0, 3.0)
    bvs_entropy_weight = trial.suggest_float('bvs_entropy_weight', 0.01, 0.3, log=True)
    bvs_min_weight = trial.suggest_float('bvs_min_weight', 0.02, 0.10)
    sum_lstm_hidden = trial.suggest_categorical('sum_lstm_hidden', [32, 64])

    # --- Transformer VAE params ---
    d_latent = trial.suggest_categorical('d_latent', [32, 64])
    d_ae_hidden = trial.suggest_categorical('d_ae_hidden', [64, 128])
    kl_weight = trial.suggest_float('kl_weight', 0.001, 0.1, log=True)
    recon_weight = trial.suggest_float('recon_weight', 0.01, 0.5, log=True)

    max_class_wt = 34.0

    # --- Dataloaders ---
    try:
        train_loader, val_loader = create_dataloaders(
            window_size=window_size, batch_size=batch_size,
        )
    except Exception as e:
        print(f"Dataloader failed: {e}")
        return -1e9

    # --- Model (MMTFAutoEncoderMamba — Transformer VAE conditioning) ---
    model = MMTFAutoEncoderMamba(
        f_sum=F_SUM,
        f_profile=F_PROFILE,
        f_raster=F_RASTER,
        f_seq=F_SEQ,
        f_ae=F_AE,
        ae_type='transformer_vae',
        d_latent=d_latent,
        d_ae_hidden=d_ae_hidden,
        kl_weight=kl_weight,
        recon_weight=recon_weight,
        n_tickers=prep.n_tickers,
        n_asset_classes=prep.n_asset_classes,
        n_asset_subclasses=prep.n_asset_subclasses,
        d_model=d_model,
        d_static_emb=d_static_emb,
        n_heads=n_attn_heads,
        n_layers=n_layers,
        d_state=d_state,
        d_conv=d_conv,
        expand=expand,
        sum_lstm_hidden=sum_lstm_hidden,
        task=TASK,
        num_classes=NUM_CLASSES,
        dropout=dropout,
        grn_dropout=grn_dropout,
        bvs_temperature=bvs_temperature,
        bvs_entropy_weight=bvs_entropy_weight,
        bvs_min_weight=bvs_min_weight,
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

    criterion = ProfitWeightedCE(
        profit_scale=profit_scale,
        min_weight=min_weight,
        max_weight=max_weight,
        direction_penalty=direction_penalty,
    ).to(device)

    best_score = -1e9
    patience_counter = 0
    prev_val_loss = None

    for epoch in range(20):
        train_loss, train_acc, train_metrics = train_epoch_vae(
            model, train_loader, criterion, optimizer, device,
            max_norm=max_norm, recon_weight=recon_weight,
        )
        val_loss, val_pnl, val_acc, val_dir_acc, val_recon, val_kl = evaluate(
            model, val_loader, criterion, device,
        )
        scheduler.step()

        if math.isnan(val_loss) or math.isinf(val_loss):
            print(f"  E{epoch+1:02d} | val_loss is NaN/Inf -- killing trial")
            return -1e9
        if val_loss > 100.0:
            print(f"  E{epoch+1:02d} | val_loss={val_loss:.1f} exploding -- killing trial")
            return -1e9
        if prev_val_loss is not None and val_loss > prev_val_loss * 5.0 and epoch >= 3:
            print(f"  E{epoch+1:02d} | val_loss spiked {prev_val_loss:.4f}->{val_loss:.4f} -- killing trial")
            return -1e9
        prev_val_loss = val_loss

        val_score = get_composite_score(val_pnl, val_acc, val_loss, baseline=max_class_wt)

        print(
            f"  E{epoch+1:02d} | Loss: {val_loss:.4f} | PnL: {val_pnl:.6f} | Score: {val_score:.4f} "
            f"| Acc: {val_acc:.1f}% | Dir: {val_dir_acc:.1f}% | Recon: {val_recon:.4f} | KL: {val_kl:.4f}"
        )

        if val_score > best_score:
            best_score = val_score
            patience_counter = 0
            trial.set_user_attr('final_pnl', val_pnl)
            trial.set_user_attr('final_score', val_score)
            trial.set_user_attr('final_acc', val_acc)
            trial.set_user_attr('final_dir_acc', val_dir_acc)
            trial.set_user_attr('final_val_loss', val_loss)
            trial.set_user_attr('final_recon_loss', val_recon)
            trial.set_user_attr('final_kl_loss', val_kl)
        else:
            patience_counter += 1

        trial.report(val_score, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        if patience_counter >= 10:
            break

    return best_score

## 5. Run Optuna Optimization

In [ ]:
N_TRIALS = 30
STUDY_NAME = f"mmtfv2_{'_'.join(TICKERS)}_{BACKBONE}_{TASK}"

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)

print(f"Starting optimization: {N_TRIALS} trials")
print(f"Study: {STUDY_NAME}")
print(f"Tickers: {', '.join(TICKERS)}")
print(f"Backbone: {BACKBONE} (fixed)")
print("-" * 50)

In [ ]:
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
    gc_after_trial=True,
)

In [ ]:
# Best trial results
best_trial = study.best_trial
print(f"\nBest trial #{best_trial.number}:")
print(f"  Score: {best_trial.value:.6f}")
if best_trial.user_attrs:
    print(f"  PnL: {best_trial.user_attrs.get('final_pnl', 'N/A')}")
    print(f"  Acc: {best_trial.user_attrs.get('final_acc', 'N/A')}")
    print(f"  Dir Acc: {best_trial.user_attrs.get('final_dir_acc', 'N/A')}")
print(f"  Params:")

best_params = best_trial.params
for key, value in sorted(best_params.items()):
    print(f"    {key}: {value}")

# Save
best_params['best_value'] = best_trial.value
best_params['tickers'] = TICKERS
best_params['task'] = TASK
best_params['backbone'] = BACKBONE

prefix = f"{'_'.join(TICKERS)}_mmtfv2_{BACKBONE}_optuna"
with open(RESULTS_PATH / f"{prefix}_best_params.json", 'w') as f:
    json.dump(best_params, f, indent=2)
print(f"\nSaved to: {RESULTS_PATH / f'{prefix}_best_params.json'}")

In [ ]:
# Save study artifacts
import joblib

joblib.dump(study, RESULTS_PATH / f"{prefix}_study.pkl")
df_trials = study.trials_dataframe()
df_trials.to_csv(RESULTS_PATH / f"{prefix}_all_trials.csv", index=False)

print(f"Study artifacts saved to {RESULTS_PATH}")

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

valid_trials = df_trials[df_trials['state'] == 'COMPLETE']

# 1. Optimization history
ax = axes[0, 0]
ax.plot(valid_trials.index, valid_trials['value'], 'b-o', alpha=0.6, label='Trial score')
ax.axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.4f}')
ax.set_xlabel('Trial')
ax.set_ylabel('Composite Score')
ax.set_title('Optimization History')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Parameter importance
ax = axes[0, 1]
try:
    importances = optuna.importance.get_param_importances(study)
    params = list(importances.keys())[:10]
    values = [importances[p] for p in params]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(params)))
    ax.barh(params, values, color=colors)
    ax.set_xlabel('Importance')
    ax.set_title('Hyperparameter Importance')
    ax.grid(True, alpha=0.3, axis='x')
except:
    ax.text(0.5, 0.5, 'Not enough completed trials', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Hyperparameter Importance')

# 3. Learning rate vs score
ax = axes[1, 0]
if 'params_learning_rate' in valid_trials.columns:
    ax.scatter(valid_trials['params_learning_rate'], valid_trials['value'],
               c=valid_trials.index, cmap='viridis', alpha=0.7, s=100)
    ax.set_xscale('log')
    ax.set_xlabel('Learning Rate')
    ax.set_ylabel('Score')
    ax.set_title('Learning Rate vs Score')
    ax.grid(True, alpha=0.3)

# 4. d_model vs score
ax = axes[1, 1]
if 'params_d_model' in valid_trials.columns:
    d_models = sorted(valid_trials['params_d_model'].unique())
    data_by_d = [valid_trials[valid_trials['params_d_model'] == d]['value'].values for d in d_models]
    bp = ax.boxplot(data_by_d, positions=range(len(d_models)), patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0, 1, len(d_models)))):
        patch.set_facecolor(color)
    ax.set_xticks(range(len(d_models)))
    ax.set_xticklabels([str(int(d)) for d in d_models])
    ax.set_xlabel('d_model')
    ax.set_ylabel('Score')
    ax.set_title('Model Size vs Score')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f"MMTFv2 Mamba Optimization ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_results.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# BVS-specific parameter analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. BVS temperature vs score
ax = axes[0, 0]
if 'params_bvs_temperature' in valid_trials.columns:
    ax.scatter(valid_trials['params_bvs_temperature'], valid_trials['value'],
               c=valid_trials.index, cmap='viridis', alpha=0.7, s=80)
    ax.set_xlabel('BVS Temperature')
    ax.set_ylabel('Score')
    ax.set_title('BVS Temperature vs Score')
    ax.grid(True, alpha=0.3)

# 2. Window size vs score
ax = axes[0, 1]
if 'params_window_size' in valid_trials.columns:
    wins = sorted(valid_trials['params_window_size'].unique())
    data_by_w = [valid_trials[valid_trials['params_window_size'] == w]['value'].values for w in wins]
    bp = ax.boxplot(data_by_w, positions=range(len(wins)), patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set3(np.linspace(0, 1, len(wins)))):
        patch.set_facecolor(color)
    ax.set_xticks(range(len(wins)))
    ax.set_xticklabels([str(int(w)) for w in wins])
    ax.set_xlabel('Window Size')
    ax.set_ylabel('Score')
    ax.set_title('Window Size vs Score')
    ax.grid(True, alpha=0.3, axis='y')

# 3. Dropout vs score
ax = axes[0, 2]
if 'params_dropout' in valid_trials.columns:
    ax.scatter(valid_trials['params_dropout'], valid_trials['value'],
               c=valid_trials.index, cmap='plasma', alpha=0.7, s=80)
    ax.set_xlabel('Dropout')
    ax.set_ylabel('Score')
    ax.set_title('Dropout vs Score')
    ax.grid(True, alpha=0.3)

# 4. BVS entropy weight vs score
ax = axes[1, 0]
if 'params_bvs_entropy_weight' in valid_trials.columns:
    ax.scatter(valid_trials['params_bvs_entropy_weight'], valid_trials['value'],
               c=valid_trials.index, cmap='coolwarm', alpha=0.7, s=80)
    ax.set_xscale('log')
    ax.set_xlabel('BVS Entropy Weight')
    ax.set_ylabel('Score')
    ax.set_title('Entropy Reg Weight vs Score')
    ax.grid(True, alpha=0.3)

# 5. Weight decay vs score
ax = axes[1, 1]
if 'params_weight_decay' in valid_trials.columns:
    ax.scatter(valid_trials['params_weight_decay'], valid_trials['value'],
               c=valid_trials.index, cmap='plasma', alpha=0.7, s=80)
    ax.set_xscale('log')
    ax.set_xlabel('Weight Decay')
    ax.set_ylabel('Score')
    ax.set_title('Weight Decay vs Score')
    ax.grid(True, alpha=0.3)

# 6. Profit scale vs score
ax = axes[1, 2]
if 'params_profit_scale' in valid_trials.columns:
    ax.scatter(valid_trials['params_profit_scale'], valid_trials['value'],
               c=valid_trials.index, cmap='coolwarm', alpha=0.7, s=80)
    ax.set_xlabel('Profit Scale')
    ax.set_ylabel('Score')
    ax.set_title('Profit Scale vs Score')
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Parameter Analysis ({', '.join(TICKERS)}, Mamba)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_param_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Train Final Model with Best Parameters

In [ ]:
best = best_params
print("Training final model with best parameters:")
for k, v in sorted(best.items()):
    if k not in ('best_value', 'tickers', 'task', 'backbone'):
        print(f"  {k}: {v}")

In [ ]:
# Create final dataloaders and model
train_loader, val_loader = create_dataloaders(
    window_size=best['window_size'],
    batch_size=best['batch_size'],
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

bvs_temperature = best.get('bvs_temperature', 1.5)
bvs_entropy_weight = best.get('bvs_entropy_weight', 0.1)
bvs_min_weight = best.get('bvs_min_weight', 0.05)
sum_lstm_hidden = best.get('sum_lstm_hidden', 64)
d_latent = best.get('d_latent', 64)
d_ae_hidden = best.get('d_ae_hidden', 128)
kl_weight = best.get('kl_weight', 0.01)
recon_weight = best.get('recon_weight', 0.1)

final_model = MMTFAutoEncoderMamba(
    f_sum=F_SUM,
    f_profile=F_PROFILE,
    f_raster=F_RASTER,
    f_seq=F_SEQ,
    f_ae=F_AE,
    ae_type='transformer_vae',
    d_latent=d_latent,
    d_ae_hidden=d_ae_hidden,
    kl_weight=kl_weight,
    recon_weight=recon_weight,
    n_tickers=prep.n_tickers,
    n_asset_classes=prep.n_asset_classes,
    n_asset_subclasses=prep.n_asset_subclasses,
    d_model=best['d_model'],
    d_static_emb=best['d_static_emb'],
    n_heads=best['n_attn_heads'],
    n_layers=best['n_layers'],
    d_state=best['d_state'],
    d_conv=best['d_conv'],
    expand=best['expand'],
    sum_lstm_hidden=sum_lstm_hidden,
    task=TASK,
    num_classes=NUM_CLASSES,
    dropout=best['dropout'],
    grn_dropout=best['grn_dropout'],
    bvs_temperature=bvs_temperature,
    bvs_entropy_weight=bvs_entropy_weight,
    bvs_min_weight=bvs_min_weight,
).to(device)

print(f"\nUsing MAMBA backbone (fixed) + Transformer VAE conditioning (ae_type='transformer_vae')")
print(f"VAE: d_latent={d_latent}, d_ae_hidden={d_ae_hidden}, kl_weight={kl_weight:.4f}, recon_weight={recon_weight:.4f}")
print(f"AE input: daily returns from intraday 10AM close (f_ae={F_AE})")
print(f"Model parameters: {sum(p.numel() for p in final_model.parameters()):,}")

In [ ]:
NUM_EPOCHS = 30

criterion = ProfitWeightedCE(
    profit_scale=best['profit_scale'],
    min_weight=best['min_weight'],
    max_weight=best['max_weight'],
    direction_penalty=best['direction_penalty'],
).to(device)

optimizer = optim.AdamW(
    final_model.parameters(),
    lr=best['learning_rate'],
    weight_decay=best['weight_decay'],
)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=best['learning_rate'] * 0.001,
)

best_max_norm = best['max_norm']

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': [],
    'val_pnl': [], 'val_dir_acc': [],
    'val_score': [], 'lr': [],
    'recon_loss': [], 'kl_loss': [],
    'train_recon': [], 'train_kl': [], 'train_entropy': [],
}

best_score = -1e9
best_state = None
max_class_wt = 34.0

print(f"Training for {NUM_EPOCHS} epochs (max_norm={best_max_norm:.2f}, MMTFAutoEncoder Mamba + VAE)...")
print(f"Loss = task_loss + {recon_weight:.3f} * (recon + {kl_weight:.4f} * KL) + entropy_reg")
print("=" * 80)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, train_metrics = train_epoch_vae(
        final_model, train_loader, criterion, optimizer, device,
        max_norm=best_max_norm, recon_weight=recon_weight,
    )
    val_loss, val_pnl, val_acc, val_dir_acc, val_recon, val_kl = evaluate(
        final_model, val_loader, criterion, device,
    )
    scheduler.step()

    val_score = get_composite_score(val_pnl, val_acc, val_loss, baseline=max_class_wt)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_pnl'].append(val_pnl)
    history['val_dir_acc'].append(val_dir_acc)
    history['val_score'].append(val_score)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['recon_loss'].append(val_recon)
    history['kl_loss'].append(val_kl)
    history['train_recon'].append(train_metrics['recon_loss'])
    history['train_kl'].append(train_metrics['kl_loss'])
    history['train_entropy'].append(train_metrics['entropy_loss'])

    is_best = val_score > best_score
    if is_best:
        best_score = val_score
        best_state = final_model.state_dict().copy()

    marker = " [BEST]" if is_best else ""
    print(
        f"E{epoch+1:02d}/{NUM_EPOCHS} | "
        f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
        f"Acc: {train_acc:.1f}/{val_acc:.1f}% | "
        f"PnL: {val_pnl:.6f} | Score: {val_score:.4f} | "
        f"Recon: {val_recon:.4f} | KL: {val_kl:.4f}{marker}"
    )

print("=" * 80)
print(f"Best composite score: {best_score:.6f}")

In [ ]:
# Load best model state
if best_state:
    final_model.load_state_dict(best_state)
    print("Loaded best model state")

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

ax = axes[0, 0]
ax.plot(history['train_loss'], label='Train', alpha=0.8)
ax.plot(history['val_loss'], label='Val', alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Task Loss'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(history['train_acc'], label='Train', alpha=0.8)
ax.plot(history['val_acc'], label='Val', alpha=0.8)
ax.axhline(y=max_class_wt, color='gray', linestyle=':', label=f'Baseline ({max_class_wt}%)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 2]
ax.plot(history['val_pnl'], 'g-', alpha=0.8, label='Avg PnL')
ax.axhline(y=0, color='gray', linestyle=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Avg PnL')
ax.set_title('Validation PnL'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(history['val_score'], 'r-', alpha=0.8, label='Composite Score')
ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
ax.set_title('Composite Score'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(history['train_recon'], label='Train Recon', alpha=0.8)
ax.plot(history['recon_loss'], label='Val Recon', alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('VAE Reconstruction Loss'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 2]
ax.plot(history['train_kl'], label='Train KL', alpha=0.8, color='purple')
ax.plot(history['kl_loss'], label='Val KL', alpha=0.8, color='orange')
ax2 = ax.twinx()
ax2.plot(history['train_entropy'], label='Entropy Reg', alpha=0.6, color='red', linestyle='--')
ax2.set_ylabel('Entropy Loss', color='red')
ax.set_xlabel('Epoch'); ax.set_ylabel('KL Loss')
ax.set_title('VAE KL + BVS Entropy')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax.grid(True, alpha=0.3)

plt.suptitle(f"MMTFAutoEncoder Mamba+VAE Training ({', '.join(TICKERS)})", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH / f"{prefix}_training_history.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Backtest Best Model

In [ ]:
from CTAFlow.models.deep_learning.training.backtest import backtest_eod_momentum

# Collect validation predictions
final_model.eval()
all_preds = []
all_targets = []
all_probs = []

with torch.no_grad():
    for batch in val_loader:
        inputs, targets = unpack_batch_for_model(batch, device=device)
        logits, _ = final_model(**inputs, return_ae_losses=True)

        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())
        all_probs.append(probs.cpu().numpy())

predictions = np.concatenate(all_preds)
targets_arr = np.concatenate(all_targets)

print(f"Predictions shape: {predictions.shape}")
print(f"Targets shape: {targets_arr.shape}")
print(f"Prediction distribution: {np.unique(predictions, return_counts=True)}")

In [ ]:
# Run backtest
proxy_returns = (targets_arr.astype(float) - 1.0) * 0.01

bt_result = backtest_eod_momentum(
    predictions=predictions,
    returns=proxy_returns,
    task='classification',
    long_class=2,
    short_class=0,
    entry_cost_bps=1.0,
    exit_cost_bps=1.0,
    entry_slippage_bps=0.5,
    exit_slippage_bps=0.5,
)

print("\nBacktest Summary:")
print("=" * 50)
for k, v in bt_result.summary.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

if hasattr(bt_result, 'trade_stats') and bt_result.trade_stats:
    print("\nTrade Statistics:")
    print("-" * 50)
    for k, v in bt_result.trade_stats.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Plot cumulative PnL
if hasattr(bt_result, 'frame') and bt_result.frame is not None:
    fig, ax = plt.subplots(figsize=(12, 5))
    cum_pnl = bt_result.frame['cum_pnl'] if 'cum_pnl' in bt_result.frame.columns else bt_result.frame['net_pnl'].cumsum()
    ax.plot(cum_pnl.values, 'b-', alpha=0.8)
    ax.axhline(y=0, color='gray', linestyle=':')
    ax.fill_between(range(len(cum_pnl)), cum_pnl.values, 0,
                    where=cum_pnl.values >= 0, color='green', alpha=0.1)
    ax.fill_between(range(len(cum_pnl)), cum_pnl.values, 0,
                    where=cum_pnl.values < 0, color='red', alpha=0.1)
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Cumulative PnL')
    ax.set_title(f'MMTFv2 Mamba Backtest: {" + ".join(TICKERS)}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_PATH / f"{prefix}_backtest_pnl.png", dpi=150, bbox_inches='tight')
    plt.show()

## 9. Model Interpretability

In [ ]:
# Collect tracker stats
final_model.eval()
branch_weights_accum = {}
static_weights_accum = {}
entropy_vals = []

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i >= 10:
            break
        inputs, _ = unpack_batch_for_model(batch, device=device)
        _, ae_losses, tracker = final_model(
            **inputs, return_ae_losses=True, return_tracker=True,
        )

        if 'branch_weights' in tracker and tracker['branch_weights']:
            for name, val in tracker['branch_weights'].items():
                branch_weights_accum.setdefault(name, []).append(val)
        if 'regime_var_weights' in tracker and tracker['regime_var_weights']:
            for name, val in tracker['regime_var_weights'].items():
                static_weights_accum.setdefault(name, []).append(val)
        if 'branch_entropy' in tracker and tracker['branch_entropy'] is not None:
            entropy_vals.append(tracker['branch_entropy'])

stats = {
    'branch_weights': {k: np.mean(v) for k, v in branch_weights_accum.items()},
    'static_var_weights': {k: np.mean(v) for k, v in static_weights_accum.items()},
    'branch_entropy': np.mean(entropy_vals) if entropy_vals else None,
}


def print_branch_diagnostics(stats):
    """Print detailed branch diagnostics including collapse detection."""
    if 'branch_weights' not in stats:
        print("No branch weights available")
        return

    bw = stats['branch_weights']
    print("Branch Importance Weights:")
    max_w = max(bw.values()) if bw else 1.0
    for name, weight in sorted(bw.items(), key=lambda x: -x[1]):
        bar = '#' * int(weight / max_w * 50) if max_w > 0 else ''
        print(f"  {name:<16s}: {weight:.4f} {bar}")

    weights = list(bw.values())
    n_active = sum(1 for w in weights if w > 0.05)
    entropy = stats.get('branch_entropy', None)

    print(f"\n  Active branches (>5%): {n_active}/{len(weights)}")
    if entropy is not None:
        print(f"  Normalized entropy: {entropy:.3f}  (0=collapsed, 1=uniform)")
        if entropy < 0.3:
            print("  WARNING: Low entropy -- branches may be collapsing")
        elif entropy > 0.7:
            print("  OK: Healthy branch diversity")

    if 'static_var_weights' in stats and stats['static_var_weights']:
        print("\nRegime Variable Weights (Transformer VAE conditioning):")
        for name, weight in sorted(stats['static_var_weights'].items(), key=lambda x: -x[1]):
            bar = '#' * int(weight * 100)
            print(f"  {name:<16s}: {weight:.4f} {bar}")


print_branch_diagnostics(stats)

## 10. Save Final Model

In [ ]:
model_path = RESULTS_PATH / f"{prefix}_best_model.pth"
torch.save({
    'model_state_dict': final_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_score': best_score,
    'params': best,
    'tickers': TICKERS,
    'dims': dims,
    'n_tickers': prep.n_tickers,
    'n_asset_classes': prep.n_asset_classes,
    'n_asset_subclasses': prep.n_asset_subclasses,
    'task': TASK,
    'num_classes': NUM_CLASSES,
    'history': history,
    'backtest_summary': bt_result.summary if bt_result else None,
    'architecture': 'MMTFAutoEncoder',
    'backbone': BACKBONE,
    'ae_type': 'transformer_vae',
    'ae_config': {
        'f_ae': F_AE,
        'd_latent': d_latent,
        'd_ae_hidden': d_ae_hidden,
        'kl_weight': kl_weight,
        'recon_weight': recon_weight,
        'ae_input_source': 'intraday.csv 10AM daily returns',
        'ae_features': ['return_1d', 'abs_return_1d', 'cumulative_5d_return'],
    },
    'scaling': {
        'summary': 'pattern_multipliers_and_rolling_zscore',
        'sequential': 'price_bps_and_orderflow_multipliers',
        'profiles': 'channel_wise_vol_div15_price_x100',
        'ae_input': 'returns_x100_clipped',
    },
}, model_path)

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_PATH / f"{prefix}_training_history.csv", index=False)

print(f"\n{'=' * 60}")
print("TRAINING COMPLETE")
print(f"{'=' * 60}")
print(f"\nArchitecture: MMTFAutoEncoder Mamba + Transformer VAE")
print(f"AE input: daily returns from intraday 10AM close (f_ae={F_AE})")
print(f"VAE config: d_latent={d_latent}, kl_weight={kl_weight:.4f}, recon_weight={recon_weight:.4f}")
print(f"\nArtifacts saved to: {RESULTS_PATH}")
print(f"  - {prefix}_best_model.pth")
print(f"  - {prefix}_best_params.json")
print(f"  - {prefix}_study.pkl")
print(f"  - {prefix}_all_trials.csv")
print(f"  - {prefix}_results.png")
print(f"  - {prefix}_training_history.png")
print(f"  - {prefix}_backtest_pnl.png")